In [ ]:
import os
import glob
import pandas as pd

kcd_base_path = '/mnt/AEGIS/AEGIS/Deidentification/AEGIS-Dx-KCD/de'

all_kcd_files = glob.glob(os.path.join(kcd_base_path, '**/*.csv'), recursive = True)

matching_pt_ids = set()

for file in all_kcd_files:
    try:
        df = pd.read_csv(file, dtype = {"pt_id":str}, sep=",", quotechar ='"', lineterminator = '\n')
        if {'depart', 'pt_id'}.issubset(df.columns):
            matched_ids = df[df['depart'].isin(['산부인과', '정신건강의학과'])]['pt_id'].dropna().unique()
            matching_pt_ids.update(matched_ids)
    except Exception as e:
        print(f"파일 처리 오류: {file} -> {e}")
        
ob_psy_pt = list(matching_pt_ids)

print(f"총 {len(ob_psy_pt)}명")

In [ ]:
import os
import glob
import pandas as pd

base_path = "/mnt/AEGIS/AEGIS/Deidentification"
target_dirs = ['AEGIS-Order-Med1',
 'AEGIS-Basic-Order',
 'AEGIS-Weight-and-Height',
 'AEGIS-Nursing-Initial-Assessment',
 'AEGIS-Dx-KCD',
 'AEGIS-Order-CIV2',
 'AEGIS-Order-Test',
 'AEGIS-Order-CIV1',
 'AEGIS-Administer-Med',
 'AEGIS-Flowsheet-Value',
 'AEGIS-Basic-Info',
 'AEGIS-Dx-SNOMED']


for folder in target_dirs:
    de_path = os.path.join(base_path, folder, 'de')

    print(f"\n📂 [{folder}] 처리 시작")

    total_before = 0
    total_after = 0

    csv_files = glob.glob(os.path.join(de_path, '**/*.csv'), recursive=True)

    for file in csv_files:
        try:
            pt_df = pd.read_csv(file, usecols = ['pt_id'], dtype={'pt_id': str}, sep=",", quotechar='"', lineterminator='\n')

            if 'pt_id' in pt_df.columns:
                
                mask = pt_df['pt_id'].isin(ob_psy_pt)
                num_to_delete = mask.sum()
                
                if num_to_delete > 0:
                    # 전체 데이터 불러오기
                    df_full = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar='"', lineterminator='\n')
                
                    before_rows = len(df_full)
                    df_full = df_full[~df_full['pt_id'].isin(ob_psy_pt)]
                    after_rows = len(df_full)

                    total_before += before_rows
                    total_after += after_rows

                    df_full.to_csv(file, index = False, encoding = 'utf-8')
                    
                    print(f"{os.path.relpath(file, de_path)} - {before_rows - after_rows}행 삭제")
                else:
                    print(f"{os.path.relpath(file, de_path)} - 변경 없음")
            else:
                print(f"{file} - pt_id 없음 - 건너뜀")
        except Exception as e:
            print(f"오류 발생: {file} -> {e}")
    print(f"{folder} 전체 행 수 변화: {total_before} -> {total_after} ({total_before - total_after}행 삭제)")

In [ ]:
ob_psy_pt = pd.DataFrame(ob_psy_pt, columns = ['pt_id'])
ob_psy_pt.to_csv('AEGIS_OB_PSY_patients.csv', index = False, encoding = 'utf-8')

In [ ]:
import os
import glob
import pandas as pd

base_path = "/mnt/AEGIS/AEGIS/Deidentification/AEGIS-Order-Med1/de"
csv_files = glob.glob(os.path.join(base_path, "**/*.csv"), recursive = True)

mapping_list = []

# med 파일 순회
for file in csv_files:
    try:
        df = pd.read_csv(file, sep=",", quotechar ='"', lineterminator = '\n')
        
        # 필수 컬럼 확인
        required_cols = {"atc_cd", "order_cd", "order_nm"}
        if required_cols.issubset(df.columns):
            filtered = df[df['atc_cd'].str.startswith('N05A', na = False)]
            mapping = filtered[['order_cd','order_nm', 'atc_cd']].drop_duplicates()
            if not mapping.empty:
                mapping_list.append(mapping)
                print(f"처리됨: {file} -> {len(mapping)}건")
            else:
                print(f"'N05A' 코드 없음: {file}")
        else:
            print(f"컬럼 없음: {file}")
            

    except Exception as e:
        print(f"오류: {file} -> {e}")
    
    
# 결과 병합
if mapping_list:
    final_mapping_df = pd.concat(mapping_list, ignore_index = True).drop_duplicates()
    print('\n 최종 매핑 결과')
    print(final_mapping_df.head())
else:
    print("N05A 관련 매핑 없음")

In [ ]:
#final_mapping_df.to_excel('N05A_mapping.xlsx', index= True)
# 다른 테이블에서 해당 처방코드 제거

del_order_codes = final_mapping_df['order_cd'].dropna().unique().tolist()
    
base_path = '/mnt/AEGIS/AEGIS/Deidentification'
target_dirs = ['AEGIS-Order-Med1',
 'AEGIS-Basic-Order',
 'AEGIS-Weight-and-Height',
 'AEGIS-Nursing-Initial-Assessment',
 'AEGIS-Dx-KCD',
 'AEGIS-Order-CIV2',
 'AEGIS-Order-Test',
 'AEGIS-Order-CIV1',
 'AEGIS-Administer-Med',
 'AEGIS-Flowsheet-Value',
 'AEGIS-Basic-Info',
 'AEGIS-Dx-SNOMED']

for folder in target_dirs:
    raw_path = os.path.join(base_path, folder, 'de')
    csv_files = glob.glob(os.path.join(raw_path, '**/*.csv'), recursive = True)
    
    print(f"\n📂 [{folder}] 처리 중...")
    
    if not csv_files:
        print(f"파일 없음: {folder}")
        continue
    
    # 첫 파일 컬럼 확인
    try:
        sample_df = pd.read_csv(csv_files[0], dtype = {"pt_id": str},nrows= 1, sep=",", quotechar ='"', lineterminator = '\n')
        if 'order_cd' not in sample_df.columns:
            print(f"'order_cd' 컬럼 없음 -> {folder} 폴더 전체 건너뜀")
            continue
    except Exception as e:
        print(f"샘플 파일 확인 오류: {csv_files[0]} -> {e}")
        raise
    
    for file in csv_files:
        try:
            df = pd.read_csv(file, dtype = {"pt_id": str}, sep=",", quotechar ='"', lineterminator = '\n')
                
            before_rows = len(df)
            df = df[~df['order_cd'].isin(del_order_codes)]                
            after_rows = len(df)
            
            if before_rows != after_rows:
                df.to_csv(file, index = False, encoding = 'utf-8')
                print(f"삭제 완료: {file} -> {before_rows - after_rows}행 제거")
            else:
                print(f"유지됨: {file}")
                
        except Exception as e:
            print(f"처리 오류: {file} -> {e}")
            raise

In [ ]:
final_mapping_df.to_csv('N05A_mapping.csv',encoding = 'utf-8', index  = False)
#pd.read_csv('N05A_mapping.csv')